# ============================================================
# STEP 2C — TURKISH LINGUISTIC FEATURES (20 features)
# Turkish Quishing Detection v10
# ============================================================
# 100% OFFLINE — extracted purely from the URL string.
# No network, no fetch, no API. Full coverage on all 1.24M rows.
#
# Novelty: Turkish character distribution, stopword density,
# n-gram language identification, and vowel-harmony heuristics
# applied to URLs — not previously used in Turkish phishing work.
#
# Input  : features_full_v10.csv  (96 features)
# Output : features_full_v11.csv  (116 features)
#          turkish_linguistic_feature_names.pkl
# ============================================================

In [ ]:
import os, re, math, pickle, warnings
from collections import Counter
from urllib.parse import urlparse, unquote
import numpy as np
import pandas as pd
import tldextract
from sklearn.feature_selection import mutual_info_classif

warnings.filterwarnings("ignore")
SEED = 42

DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
INPUT_FILE  = os.path.join(DATA_DIR, "features_full_final.csv")
OUTPUT_CSV  = os.path.join(DATA_DIR, "features_full_final.csv")
OUTPUT_PKL  = os.path.join(DATA_DIR, "turkish_linguistic_feature_names_final.pkl")

# ════════════════════════════════════════════════════════════
# TURKISH LANGUAGE RESOURCES
# ════════════════════════════════════════════════════════════
TR_CHARS = set("çğıİöşüÇĞIÖŞÜ")  # Turkish-specific characters
TR_VOWELS = set("aeıioöuü")
TR_FRONT_VOWELS = set("eiöü")
TR_BACK_VOWELS  = set("aıou")

# Common Turkish stopwords (appear in legitimate Turkish URLs/paths)
TR_STOPWORDS = {
    "ve","ile","için","bu","bir","da","de","ki","mi","ne",
    "ya","veya","ama","fakat","çünkü","gibi","kadar","daha",
    "en","çok","az","her","hep","tüm","bütün","olan","olarak",
    "ana","sayfa","hakkinda","hakkında","iletisim","iletişim",
    "urunler","ürünler","hizmetler","kategori","detay","liste",
}

# Turkish derivational/inflectional suffixes (strong Turkish signal)
TR_SUFFIXES = [
    "lar","ler","dan","den","tan","ten","nin","nın","nun","nün",
    "da","de","ta","te","ya","ye","yi","yı","yu","yü",
    "lik","lık","luk","lük","ci","cı","cu","cü","li","lı","lu","lü",
    "siz","sız","suz","süz"," size","sını","ları","leri",
]

# Turkish sector terminology (domain-level)
TR_BANK_TERMS = ["banka","bank","kredi","hesap","odeme","ödeme","havale",
                 "iban","kart","finans","yatirim","yatırım","sigorta"]
TR_GOV_TERMS  = ["devlet","gov","belediye","valilik","kaymakamlik",
                 "bakanlik","bakanlık","mahkeme","emniyet","sgk","vergi",
                 "nufus","nüfus","kimlik","pasaport","ehliyet"]
TR_ECOM_TERMS = ["magaza","mağaza","alisveris","alışveriş","sepet",
                 "indirim","kampanya","urun","ürün","siparis","sipariş",
                 "kargo","fatura","odeme","market"]
TR_TELECOM_TERMS = ["turkcell","vodafone","turktelekom","türktelekom",
                    "ttnet","superonline","fatura","tarife","paket",
                    "kontör","kontor","fatura"]

# Turkish character bigram log-frequencies (from Turkish corpus statistics)
# High-frequency Turkish bigrams that rarely appear in English
TR_COMMON_BIGRAMS = {
    "ar","an","la","er","in","le","de","ya","ki","ne","ri","ma",
    "il","ka","ta","ba","sa","da","ra","ek","at","el","li","ge",
    "ye","na","te","me","ni","im","ol","es","ün","ır","iş","ış",
}
EN_COMMON_BIGRAMS = {
    "th","he","in","er","an","re","on","at","en","nd","ti","es",
    "or","te","of","ed","is","it","al","ar","st","to","nt","ng",
}

# ════════════════════════════════════════════════════════════
# FEATURE NAMES
# ════════════════════════════════════════════════════════════
TR_LINGUISTIC_FEATURES = [
    # Group A: character distribution (6)
    "tr_char_count","tr_char_ratio","has_tr_char",
    "tr_char_in_domain","tr_char_in_path","distinct_tr_chars",
    # Group B: stopword / morphology (5)
    "tr_stopword_count","tr_stopword_ratio","tr_suffix_count",
    "vowel_harmony_score","tr_token_ratio",
    # Group C: n-gram language ID (4)
    "tr_bigram_score","tr_vs_en_bigram","langid_tr_confidence",
    "is_turkish_dominant",
    # Group D: sector terminology (5)
    "has_tr_bank_term","has_tr_gov_term","has_tr_ecommerce_term",
    "has_tr_telecom_term","tr_sector_count",
]
assert len(TR_LINGUISTIC_FEATURES) == 20

# ════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════
def tokenize(url):
    """Split URL into word-like tokens."""
    u = unquote(str(url).lower())
    return [t for t in re.split(r"[^a-zçğıöşü0-9]+", u) if len(t) >= 2]

def vowel_harmony(token):
    """
    Turkish vowel harmony: a word tends to use either all front
    or all back vowels. Returns 1.0 for perfect harmony, lower for mixed.
    Legitimate Turkish words score high; random strings score low.
    """
    vowels = [c for c in token if c in TR_VOWELS]
    if len(vowels) < 2:
        return 0.0
    front = sum(1 for v in vowels if v in TR_FRONT_VOWELS)
    back  = sum(1 for v in vowels if v in TR_BACK_VOWELS)
    return max(front, back) / len(vowels)

# ════════════════════════════════════════════════════════════
# EXTRACT
# ════════════════════════════════════════════════════════════
def extract_turkish_linguistic(url):
    u = unquote(str(url).lower())
    ext = tldextract.extract(url if str(url).startswith("http") else "http://" + str(url))
    try:
        parsed = urlparse(url if str(url).startswith("http") else "http://" + str(url))
        path = parsed.path or ""
    except Exception:
        path = ""
    domain = (ext.domain or "")
    n = max(len(u), 1)
    f = {}

    # ── Group A: Turkish character distribution ──────────────
    tr_chars = [c for c in u if c in TR_CHARS]
    f["tr_char_count"]    = len(tr_chars)
    f["tr_char_ratio"]    = len(tr_chars) / n
    f["has_tr_char"]      = int(len(tr_chars) > 0)
    f["tr_char_in_domain"] = int(any(c in TR_CHARS for c in domain))
    f["tr_char_in_path"]   = int(any(c in TR_CHARS for c in path.lower()))
    f["distinct_tr_chars"] = len(set(tr_chars))

    # ── Group B: stopwords / morphology ──────────────────────
    tokens = tokenize(u)
    n_tok = max(len(tokens), 1)
    sw = sum(1 for t in tokens if t in TR_STOPWORDS)
    f["tr_stopword_count"] = sw
    f["tr_stopword_ratio"] = sw / n_tok
    f["tr_suffix_count"]   = sum(
        1 for t in tokens if any(t.endswith(s) for s in TR_SUFFIXES) and len(t) > 4)
    harmony = [vowel_harmony(t) for t in tokens if len(t) >= 4]
    f["vowel_harmony_score"] = round(np.mean(harmony), 4) if harmony else 0.0
    # token "looks Turkish": has tr char, suffix, or harmony > 0.8
    tr_like = sum(1 for t in tokens
                  if any(c in TR_CHARS for c in t)
                  or any(t.endswith(s) for s in TR_SUFFIXES)
                  or vowel_harmony(t) > 0.8)
    f["tr_token_ratio"] = tr_like / n_tok

    # ── Group C: n-gram language ID ──────────────────────────
    alpha = re.sub(r"[^a-zçğıöşü]", "", u)
    bigrams = [alpha[i:i+2] for i in range(len(alpha)-1)]
    nb = max(len(bigrams), 1)
    tr_bg = sum(1 for b in bigrams if b in TR_COMMON_BIGRAMS)
    en_bg = sum(1 for b in bigrams if b in EN_COMMON_BIGRAMS)
    f["tr_bigram_score"] = tr_bg / nb
    f["tr_vs_en_bigram"] = (tr_bg - en_bg) / nb
    # Confidence: tr bigram dominance + tr chars + suffixes
    conf = (tr_bg / nb) * 0.5 + f["tr_char_ratio"] * 3 + \
           (f["tr_suffix_count"] / n_tok) * 0.5
    f["langid_tr_confidence"] = round(min(conf, 1.0), 4)
    f["is_turkish_dominant"] = int(
        f["langid_tr_confidence"] > 0.15 or f["tr_char_ratio"] > 0.02)

    # ── Group D: sector terminology ──────────────────────────
    f["has_tr_bank_term"]      = int(any(t in u for t in TR_BANK_TERMS))
    f["has_tr_gov_term"]       = int(any(t in u for t in TR_GOV_TERMS))
    f["has_tr_ecommerce_term"] = int(any(t in u for t in TR_ECOM_TERMS))
    f["has_tr_telecom_term"]   = int(any(t in u for t in TR_TELECOM_TERMS))
    f["tr_sector_count"] = (f["has_tr_bank_term"] + f["has_tr_gov_term"] +
                            f["has_tr_ecommerce_term"] + f["has_tr_telecom_term"])

    return {k: f[k] for k in TR_LINGUISTIC_FEATURES}


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
print("=" * 70)
print("STEP 2C — TURKISH LINGUISTIC FEATURES (20 features)")
print("=" * 70)

df = pd.read_csv(INPUT_FILE, low_memory=False)
print(f"\n[1] Loaded {len(df):,} rows, {len(df.columns)} cols")

# Validation on examples
print(f"\n[2] Validation on examples:")
tests = [
    "https://garanti.com.tr/bireysel/krediler/konut-kredisi-basvurusu",
    "http://garanti-giris.xyz/hesap-dogrulama",
    "https://www.sahibinden.com/ilan/emlak-konut-satilik",
    "http://osbases.tribun-triptych.lat/k5s8-byna-tqed",
    "https://e-devlet.gov.tr/vatandas/giris",
]
for t in tests:
    ft = extract_turkish_linguistic(t)
    print(f"  {t[:52]}")
    print(f"    tr_conf={ft['langid_tr_confidence']:.3f} "
          f"harmony={ft['vowel_harmony_score']:.2f} "
          f"sectors={ft['tr_sector_count']} "
          f"tr_dom={ft['is_turkish_dominant']}")

# Extract all
print(f"\n[3] Extracting for {len(df):,} URLs (offline, ~3-5 min) ...")
records = []
for i, url in enumerate(df["url"]):
    records.append(extract_turkish_linguistic(url))
    if (i + 1) % 100_000 == 0:
        print(f"    {i+1:>9,} / {len(df):,}")

tr_df = pd.DataFrame(records)
print(f"\n[4] Matrix: {tr_df.shape}  Nulls: {tr_df.isnull().sum().sum()}")

# Merge
META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
EXISTING = [c for c in df.columns if c not in META]
full = pd.concat([df[EXISTING].reset_index(drop=True),
                  tr_df.reset_index(drop=True)], axis=1)
for c in ["url","source","tld","label","label_enc",
          "class_final","class_enc","fold","reg_domain"]:
    if c in df.columns:
        full[c] = df[c].values
full.to_csv(OUTPUT_CSV, index=False)
with open(OUTPUT_PKL, "wb") as fh:
    pickle.dump(TR_LINGUISTIC_FEATURES, fh)
print(f"\n[5] Saved {len(EXISTING)+20} features → {OUTPUT_CSV}")

# Class rates
print(f"\n[6] Turkish feature rates by class:")
tr_df["class_final"] = df["class_final"].values
key = ["has_tr_char","is_turkish_dominant","tr_sector_count",
       "vowel_harmony_score","langid_tr_confidence","has_tr_bank_term"]
print(tr_df.groupby("class_final")[key].mean().round(3).to_string())

# MI
print(f"\n[7] MI ranking — Turkish features (5-class):")
X = tr_df[TR_LINGUISTIC_FEATURES].values
mi = mutual_info_classif(X, df["class_enc"].values, random_state=SEED)
mi_s = pd.Series(mi, index=TR_LINGUISTIC_FEATURES).sort_values(ascending=False)
print(mi_s.head(10).round(4).to_string())

print(f"""
{'='*70}
STEP 2C COMPLETE — 20 Turkish linguistic features (116 total)
{'='*70}
  Top Turkish feature (5-class): {mi_s.index[0]} (MI={mi_s.iloc[0]:.4f})
  Output: {OUTPUT_CSV}
  Next → Step 2D: graph-based infrastructure features
{'='*70}
""")

STEP 2C — TURKISH LINGUISTIC FEATURES (20 features)

[1] Loaded 1,239,308 rows, 105 cols

[2] Validation on examples:
  https://garanti.com.tr/bireysel/krediler/konut-kredi
    tr_conf=0.131 harmony=0.81 sectors=1 tr_dom=0
  http://garanti-giris.xyz/hesap-dogrulama
    tr_conf=0.125 harmony=0.63 sectors=1 tr_dom=0
  https://www.sahibinden.com/ilan/emlak-konut-satilik
    tr_conf=0.271 harmony=0.57 sectors=0 tr_dom=1
  http://osbases.tribun-triptych.lat/k5s8-byna-tqed
    tr_conf=0.108 harmony=0.17 sectors=0 tr_dom=0
  https://e-devlet.gov.tr/vatandas/giris
    tr_conf=0.121 harmony=0.75 sectors=1 tr_dom=0

[3] Extracting for 1,239,308 URLs (offline, ~3-5 min) ...
      100,000 / 1,239,308
      200,000 / 1,239,308
      300,000 / 1,239,308
      400,000 / 1,239,308
      500,000 / 1,239,308
      600,000 / 1,239,308
      700,000 / 1,239,308
      800,000 / 1,239,308
      900,000 / 1,239,308
    1,000,000 / 1,239,308
    1,100,000 / 1,239,308
    1,200,000 / 1,239,308

[4] Matrix: (12